# Use the orchestration service of Generative AI Hub

The orchestration service of Generative AI Hub lets you use all the available models with the same codebase. You only deploy the orchestration service and then you can access all available models simply by changing the model name parameter. You can also use grounding, prompt templating, data masking and content filtering capabilities.

In [ ]:
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

AICORE_ORCHESTRATION_DEPLOYMENT_URL = "https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/D276250e074fc028"

### Import the packages you want to use

In [ ]:
from gen_ai_hub.orchestration.models.llm import LLM
from gen_ai_hub.orchestration.models.config import GroundingModule, OrchestrationConfig
from gen_ai_hub.orchestration.models.document_grounding import DocumentGrounding, DocumentGroundingFilter, DataRepositoryType, GroundingFilterSearch
from gen_ai_hub.orchestration.models.template import Template, TemplateValue
from gen_ai_hub.orchestration.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration.service import OrchestrationService

In [ ]:
llm = LLM(
    name="gemini-2.5-flash",
    parameters={
        'temperature': 0.0,
    }
)
template = Template(
            messages=[
                SystemMessage("You are a helpful translation assistant."),
                UserMessage("""Answer the request by providing relevant answers that fit to the request.
                Request: {{ ?user_query }}
                Context:{{ ?grounding_response }}
                """),
            ]
        )

### Create an orchestration configuration that specifies the grounding capability

In [ ]:
# Set up Document Grounding
filters = [
            DocumentGroundingFilter(
                id="vector", data_repositories=["bb5bd2b6-299d-44d4-9c00-3a248bee03a3"],
                search_config=GroundingFilterSearch(max_chunk_count=15),
                data_repository_type=DataRepositoryType.VECTOR.value
            )
        ]

grounding_config = GroundingModule(
            type="document_grounding_service",
            config=DocumentGrounding(input_params=["user_query"], output_param="grounding_response", filters=filters)
        )

config = OrchestrationConfig(
    template=template,
    llm=llm,
    grounding=grounding_config
)

In [ ]:
orchestration_service = OrchestrationService(
    api_url=AICORE_ORCHESTRATION_DEPLOYMENT_URL,
    config=config
)

response = orchestration_service.stream(
    template_values=[
        TemplateValue(
            name="user_query",
            #TODO Here you can change the user prompt into whatever you want to ask the model
            value="What is AWS Nova?"
        )
    ]
)

for chunk in response:
    print(chunk.orchestration_result.choices[0].delta.content)